# HCMT — Notebook 03: Attention Map Visualization & Interpretability

This notebook visualizes:
1. Cross-modal attention heatmaps (which modalities attend to each other)
2. Per-sample gate weight distributions
3. Average modality contributions per PAM50 subtype
4. Confusion matrix analysis
5. WSI patch attention overlays

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models.hcmt import HCMTClassifier, PAM50_CLASSES
from src.data.dataset import build_dataloaders
from src.utils.utils import load_config, load_checkpoint
from src.visualization.attention_viz import (
    visualize_cross_modal_attention,
    visualize_gate_weights,
    visualize_average_gate_weights_by_subtype,
    plot_confusion_matrix,
    plot_training_history,
)
from src.evaluation.metrics import compute_metrics

sns.set_style('whitegrid')
%matplotlib inline

CHECKPOINT = '../checkpoints/hcmt_baseline_best.pth'
CONFIG     = '../configs/default.yaml'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Model

In [ ]:
cfg = load_config(CONFIG)
model = HCMTClassifier.from_config(cfg)
ckpt = load_checkpoint(model, CHECKPOINT, device=str(device))
model = model.to(device).eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')
print(f'Val metrics: {ckpt["metrics"]}')

## 2. Run Inference on Test Set

In [ ]:
_, _, test_loader, _ = build_dataloaders(cfg)

all_logits, all_labels, all_gate_w, all_attn = [], [], [], []

with torch.no_grad():
    for batch in test_loader:
        wsi       = batch['wsi'].to(device)       if batch['wsi']       is not None else None
        genomics  = batch['genomics'].to(device)  if batch['genomics']  is not None else None
        radiology = batch['radiology'].to(device) if batch['radiology'] is not None else None
        clinical  = batch['clinical'].to(device)  if batch['clinical']  is not None else None
        available = {k: v.to(device) for k, v in batch['available'].items()}

        logits, gate_w, attn = model(wsi, genomics, radiology, clinical, available)
        all_logits.append(logits.cpu())
        all_labels.append(batch['label'])
        all_gate_w.append(gate_w.cpu())
        all_attn.append({k: v.cpu() if v is not None else None for k, v in attn.items()})

logits     = torch.cat(all_logits)
labels     = torch.cat(all_labels)
gate_w     = torch.cat(all_gate_w)

print(f'Test set: {len(labels)} samples')
metrics = compute_metrics(logits, labels)
print(f'Macro F1:          {metrics["macro_f1"]:.4f}')
print(f'Balanced Accuracy: {metrics["balanced_accuracy"]:.4f}')
print(f'Mean AUC:          {metrics["mean_auc"]:.4f}')

## 3. Confusion Matrix

In [ ]:
fig = plot_confusion_matrix(
    metrics['confusion_matrix'],
    normalize=True,
    title='HCMT — PAM50 Subtype Confusion Matrix'
)
plt.show()

## 4. Cross-Modal Attention Heatmap

In [ ]:
# Aggregate attention weights across all batches
agg_attn = {}
for k in all_attn[0]:
    parts = [b[k] for b in all_attn if b.get(k) is not None]
    if parts:
        agg_attn[k] = torch.cat(parts, dim=0)

fig = visualize_cross_modal_attention(
    agg_attn,
    title='HCMT — Cross-Modal Attention (Test Set Average)'
)
plt.show()

print('Interpretation:')
print('  Row = Query modality (the one being updated)')
print('  Col = Context modality (the one being attended to)')
print('  High value = Row modality heavily attends to Col modality')

## 5. Modality Gate Weights by PAM50 Subtype

In [ ]:
fig = visualize_average_gate_weights_by_subtype(gate_w, labels)
plt.show()

print('\nInterpretation:')
print('  Which modalities does the model rely on most for each subtype?')
print('  High genomics weight for HER2 → genomic amplification is key')
print('  High WSI weight for Basal → morphological features are discriminative')

## 6. Per-Sample Gate Weights (Sample of Patients)

In [ ]:
# Show 30 random test patients
n_show = 30
idx = torch.randperm(len(labels))[:n_show]

fig = visualize_gate_weights(
    gate_w[idx],
    true_labels=labels[idx],
)
plt.show()

## 7. Per-Class Metric Summary

In [ ]:
import pandas as pd

rows = []
for cls in PAM50_CLASSES:
    rows.append({
        'Subtype':    cls,
        'Precision':  metrics.get(f'precision_{cls}', 0),
        'Recall':     metrics.get(f'recall_{cls}', 0),
        'F1':         metrics.get(f'f1_{cls}', 0),
        'AUC':        metrics.get(f'auc_{cls}', float('nan')),
        'Support':    metrics.get(f'support_{cls}', 0),
    })

df = pd.DataFrame(rows).set_index('Subtype')
print(df.to_string(float_format=lambda x: f'{x:.4f}'))

# Bar chart of per-class F1
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#6366f1', '#10b981', '#f59e0b', '#ef4444', '#94a3b8']
ax.bar(df.index, df['F1'], color=colors, alpha=0.85, edgecolor='white')
ax.axhline(metrics['macro_f1'], color='black', linestyle='--',
           label=f'Macro F1: {metrics["macro_f1"]:.4f}')
ax.set_ylim(0, 1)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Scores', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()